# Monte Carlo uncertainty, explained line by line

This notebook explains the **exact** method this project uses to put an error bar on a
predicted lattice thermal conductivity. It does not describe a simplified version — it
imports and runs `scripts/cgcnn/07_predict_kappa.py`'s own physics, and at the end it
checks that the code written here reproduces that script's output exactly.

**Run it with the project environment:**
`/Users/mac/miniconda3/envs/ml_env/bin/jupyter-lab`, kernel `ml_env`.

---

## The problem, in one paragraph

We predict two elastic moduli, $K$ and $G$, then push them through a chain of physics
formulas to get $\kappa_L$. Each formula is exact. But $K$ and $G$ are **predictions**, and
predictions are uncertain. So $\kappa_L$ is uncertain too.

The question Monte Carlo answers: **given that $K$ and $G$ are uncertain by a known amount,
how uncertain is $\kappa_L$?**

## Why not just propagate the error algebraically?

The classical approach is the *delta method*: differentiate $\kappa_L$ with respect to $K$
and $G$, and combine the partial derivatives. That works when the function is close to
linear over the range of the uncertainty. Ours is not:

$$\kappa_L = \frac{G\,v_s\,V^{1/3}}{N\,T}\,e^{-\gamma}, \qquad
  \gamma = \frac{3(1+\nu)}{2(2-3\nu)}, \qquad
  \nu = \frac{(v_l/v_t)^2-2}{2(v_l/v_t)^2-2}$$

$\gamma$ sits in an **exponential**, and $\nu$ has $K$ and $G$ inside a ratio inside a
rational function. Linearising that is a bad approximation, and the algebra is unpleasant.

Monte Carlo sidesteps both problems: instead of differentiating the function, we **feed it
many possible inputs and look at the spread of outputs**. No derivatives, no linearity
assumption, and it is exact in the limit of many samples.

---
## 1 · Setup

Nothing here is specific to Monte Carlo — this cell only makes the project's own code
importable so we can use the *real* physics rather than a copy.

In [ ]:
# torch FIRST. This project's environment loads Intel MKL, which brings its own
# OpenMP runtime; if numpy/pandas are imported first, a second copy of
# libiomp5 gets loaded and the process aborts. Importing torch first wins the
# race. (07_predict_kappa.py imports numpy only, but we follow the house rule.)
import os, sys, math
import torch  # noqa: F401  <- MUST come before numpy/pandas, see the comment above

import numpy as np                 # array maths; the whole Monte Carlo is vectorised numpy
import pandas as pd                # only to read the predictions CSV
import matplotlib.pyplot as plt    # the plots further down

# The project root is one level up from notebooks/. Adding it (and scripts/cgcnn)
# to sys.path lets us import the real physics function instead of retyping it.
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "scripts", "cgcnn"))

from importlib import import_module
# 07_predict_kappa starts with a digit, so it cannot be imported with the normal
# `import` statement (identifiers cannot start with a digit) - import_module takes
# the name as a string and sidesteps that.
_kap = import_module("07_predict_kappa")
slack_physics = _kap.slack_physics       # THE function under discussion
monte_carlo_kappa = _kap.monte_carlo_kappa   # the project's own MC, for the check at the end

print("imported slack_physics from", _kap.__file__)

---
## 2 · What the ensemble hands us

The moduli come from an **ensemble**: three networks trained from different random
initialisations. For each crystal they give three slightly different answers.

- their **mean** (in log space) is the prediction
- their **standard deviation** is the uncertainty

That standard deviation is the only input the Monte Carlo needs. It is already stored in
the predictions file as `K_VRH_spread_log10` and `G_VRH_spread_log10`.

Note **`_log10`** in the names — the spread is measured on $\log_{10}$ of the modulus, not
on the modulus in GPa. Section 5 explains why that matters.

In [ ]:
# Load the moduli predictions produced by 04_predict_moduli.py / 05_ensemble.py.
pred = pd.read_csv(os.path.join(ROOT, "results", "cgcnn", "pink_moduli_predictions.csv"))

# The five columns the Monte Carlo actually consumes:
cols = ["formula", "K_VRH_pred", "K_VRH_spread_log10", "G_VRH_pred", "G_VRH_spread_log10"]
print(f"{len(pred)} crystals\n")
print(pred[cols].head(5).to_string(index=False))

print("\nHow big is the ensemble disagreement, typically?")
print(f"  K spread (log10): median {pred.K_VRH_spread_log10.median():.4f}   "
      f"90th pct {pred.K_VRH_spread_log10.quantile(0.9):.4f}")
print(f"  G spread (log10): median {pred.G_VRH_spread_log10.median():.4f}   "
      f"90th pct {pred.G_VRH_spread_log10.quantile(0.9):.4f}")

# A log10 spread of s means a multiplicative uncertainty of 10**s. Translating
# makes it concrete: 0.02 in log10 is about a 4.7% uncertainty in GPa.
s = pred.K_VRH_spread_log10.median()
print(f"\n  a log10 spread of {s:.4f} means a factor of 10^{s:.4f} = {10**s:.4f},")
print(f"  i.e. roughly +/- {100*(10**s - 1):.1f}% on the modulus in GPa")

---
## 3 · The method, on ONE crystal

Take a single crystal and do the whole thing by hand, so every step is visible.

The recipe has three steps:

1. **Draw** many plausible $(K, G)$ pairs, consistent with the ensemble's uncertainty
2. **Push** every pair through the physics — the identical function used for the point estimate
3. **Summarise** the resulting spread of $\kappa_L$ values with percentiles

In [ ]:
# --- pick one crystal to follow all the way through -----------------------
row = pred.iloc[0]
print(f"crystal: {row.formula}")
print(f"  K = {row.K_VRH_pred:8.3f} GPa   ensemble spread {row.K_VRH_spread_log10:.4f} (log10)")
print(f"  G = {row.G_VRH_pred:8.3f} GPa   ensemble spread {row.G_VRH_spread_log10:.4f} (log10)")

# The structural quantities come from the CIF and are NOT uncertain - they are
# measured facts about the crystal, not predictions. Only K and G get sampled.
kappa_inputs = pd.read_csv(os.path.join(ROOT, "results", "cgcnn", "pink_kappa_predictions.csv"))
srow = kappa_inputs[kappa_inputs.material_id == row.material_id].iloc[0]
V   = float(srow["Volume (A3)"])          # unit-cell volume, cubic angstroms
RHO = float(srow["Density (g cm-3)"])     # density
MBAR= float(srow["Atomic mass (amu)"])    # mean atomic mass
N   = float(srow["Number of Atoms"])      # atoms in the cell
print(f"  structure (fixed): V={V:.1f} A^3, rho={RHO:.3f} g/cm3, Mbar={MBAR:.1f} amu, N={N:.0f}")

### Step 1 — draw the samples

For each sample we draw a random $\log_{10}K$ from a normal distribution centred on the
prediction, with standard deviation equal to the ensemble spread:

$$\log_{10}K^{(i)} \sim \mathcal{N}\!\left(\log_{10}K_{\text{pred}},\; \sigma_K^2\right)$$

and the same independently for $G$.

**Why normal?** The ensemble members scatter around their mean roughly symmetrically in log
space, and with only three members we have no evidence for a more complicated shape. A
normal is the least-assuming choice given a mean and a spread.

**Why independent?** The $K$ ensemble and the $G$ ensemble are separately seeded runs that
share no weights and no random state, so a large error on $K$ carries no information about
the error on $G$. *(This is an assumption worth flagging — in reality both models see the
same crystal and could fail on it together. Section 7 returns to it.)*

In [ ]:
N_SAMPLES = 2000          # the project's default (--mc-samples); Section 6 shows why 2000
SEED      = 0             # fixed so the notebook is reproducible run to run

rng = np.random.RandomState(SEED)   # a seeded generator: same seed -> same numbers, always

# standard_normal gives draws from N(0, 1). Scaling by the spread and adding the
# mean turns them into draws from N(mean, spread^2) - this is the standard
# "location-scale" trick, and it is exactly what 07_predict_kappa.py does.
z_k = rng.standard_normal(N_SAMPLES)     # 2000 draws from N(0,1) for K
z_g = rng.standard_normal(N_SAMPLES)     # 2000 independent draws for G

log_k = np.log10(row.K_VRH_pred) + row.K_VRH_spread_log10 * z_k   # N(log10 K_pred, sigma_K)
log_g = np.log10(row.G_VRH_pred) + row.G_VRH_spread_log10 * z_g   # N(log10 G_pred, sigma_G)

# The physics wants GPa, not logs, so undo the log10.
k_samples = 10 ** log_k    # 2000 plausible bulk moduli, in GPa
g_samples = 10 ** log_g    # 2000 plausible shear moduli, in GPa

print(f"drew {N_SAMPLES} (K, G) pairs")
print(f"  K samples: min {k_samples.min():7.2f}  median {np.median(k_samples):7.2f}  max {k_samples.max():7.2f} GPa")
print(f"  G samples: min {g_samples.min():7.2f}  median {np.median(g_samples):7.2f}  max {g_samples.max():7.2f} GPa")

### Step 2 — push every sample through the physics

This is the step that makes Monte Carlo work. We do **not** approximate the function; we
evaluate it, 2000 times, on inputs that represent our uncertainty.

`slack_physics` is written with numpy broadcasting, so passing arrays instead of scalars
runs all 2000 at once — no Python loop.

In [ ]:
# slack_physics(k_gpa, g_gpa, volume_a3, density, mass_amu, n_atoms).
# k_samples and g_samples are arrays of length 2000; the four structural values
# are plain floats. numpy broadcasts the floats against the arrays, so we get
# 2000 kappa values out - one per sampled (K, G) pair.
out = slack_physics(k_samples, g_samples, V, RHO, MBAR, N)

kappa_samples = out["kappa_cal"]        # the quantity this project reports
gamma_samples = out["gruneisen"]        # carried along to show how it varies too

print(f"got {kappa_samples.shape[0]} kappa values out")
print(f"  kappa: min {kappa_samples.min():.4f}   median {np.median(kappa_samples):.4f}   "
      f"max {kappa_samples.max():.4f}  W/m/K")
print(f"  gamma: min {gamma_samples.min():.4f}   median {np.median(gamma_samples):.4f}   "
      f"max {gamma_samples.max():.4f}")

# The point estimate: the SAME function on the unsampled predictions.
point = slack_physics(np.array([row.K_VRH_pred]), np.array([row.G_VRH_pred]),
                      V, RHO, MBAR, N)["kappa_cal"][0]
print(f"\npoint estimate (no sampling): {point:.4f} W/m/K")

### Step 3 — summarise with percentiles

We now have 2000 values of $\kappa_L$. The **5th and 95th percentiles** bracket the middle
90% of them, which is what this project reports as `Kappa_cal_p05` and `Kappa_cal_p95`.

**Why percentiles and not mean ± 2 sd?** Because the output distribution is *not* normal
even though the input was. The physics is non-linear — it squashes and stretches the
distribution — so a symmetric interval around the mean would misrepresent it. Percentiles
make no shape assumption at all: they just say "5% of the samples fell below this value".

In [ ]:
p05, p50, p95 = np.percentile(kappa_samples, [5, 50, 95])

print(f"  p05 (5th percentile)  {p05:.4f} W/m/K")
print(f"  p50 (median)          {p50:.4f}")
print(f"  p95 (95th percentile) {p95:.4f}")
print(f"\n  the interval spans a factor of {p95/p05:.2f}")
print(f"  point estimate {point:.4f} sits at the "
      f"{100*(kappa_samples < point).mean():.0f}th percentile of the samples")

# Is the output distribution symmetric? Compare the two half-widths.
print(f"\n  distance from median down to p05: {p50 - p05:.4f}")
print(f"  distance from median up   to p95: {p95 - p50:.4f}")
print("  -> not equal, so the distribution is skewed and 'mean +/- 2sd' would be wrong")

### The picture

Left: the sampled inputs. Right: the sampled outputs. Watch the shape change — a symmetric
input distribution comes out **skewed**, which is exactly why percentiles are used.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8))

# --- input: sampled bulk modulus ---
ax = axes[0]
ax.hist(k_samples, bins=60, color="#1450AA", edgecolor="white", lw=0.3)
ax.axvline(row.K_VRH_pred, color="#D97B12", lw=2, label="prediction")
ax.set_xlabel("sampled $K$ (GPa)"); ax.set_ylabel("samples")
ax.set_title("A  input: 2000 draws of $K$", fontsize=10)
ax.legend(fontsize=8, frameon=False)

# --- the derived Gruneisen parameter ---
ax = axes[1]
ax.hist(gamma_samples, bins=60, color="#1E8E5A", edgecolor="white", lw=0.3)
ax.set_xlabel(r"derived $\gamma$")
ax.set_title(r"B  $\gamma$ is a function of $K$ and $G$" "\n" "so it inherits their spread", fontsize=10)

# --- output: kappa, with the percentile interval marked ---
ax = axes[2]
ax.hist(kappa_samples, bins=60, color="#7A3FA0", edgecolor="white", lw=0.3)
for v, c, lbl in [(p05, "#D97B12", "p05"), (p50, "#1F2937", "median"), (p95, "#D97B12", "p95")]:
    ax.axvline(v, color=c, lw=2, ls="--" if lbl != "median" else "-")
    ax.text(v, ax.get_ylim()[1]*0.92, lbl, rotation=90, fontsize=8,
            ha="right", va="top", color=c)
ax.set_xlabel(r"sampled $\kappa_L$ (W m$^{-1}$K$^{-1}$)")
ax.set_title("C  output: skewed, despite a\nsymmetric input", fontsize=10)

for a in axes:
    for s_ in ("top", "right"): a.spines[s_].set_visible(False)
    a.grid(color="#E1E6EF", lw=0.7); a.set_axisbelow(True)
plt.tight_layout(); plt.show()

---
## 4 · Doing all 1,213 crystals at once

The single-crystal version above uses 1-D arrays. The project's version uses **2-D** arrays
of shape `(n_crystals, n_samples)` so every crystal is sampled simultaneously.

The only new idea is *broadcasting*: `values[:, None]` turns a length-1213 array into a
1213×1 column, which numpy then stretches across the 2000 sample columns.

In [ ]:
n = len(pred)
rng = np.random.RandomState(SEED)             # fresh generator, same seed

# shape (n_crystals, n_samples): one row per crystal, one column per sample
z_k = rng.standard_normal((n, N_SAMPLES))
z_g = rng.standard_normal((n, N_SAMPLES))

# [:, None] reshapes (n,) -> (n, 1). numpy then broadcasts that single column
# across all 2000 sample columns, so each crystal gets its OWN mean and spread.
log_k = np.log10(pred.K_VRH_pred.values)[:, None] + pred.K_VRH_spread_log10.values[:, None] * z_k
log_g = np.log10(pred.G_VRH_pred.values)[:, None] + pred.G_VRH_spread_log10.values[:, None] * z_g

print(f"z_k shape          {z_k.shape}   (crystals x samples)")
print(f"mean column shape  {pred.K_VRH_pred.values[:, None].shape}  -> broadcast across samples")
print(f"log_k shape        {log_k.shape}")
print(f"\nthat is {log_k.size:,} sampled moduli, evaluated in one vectorised call")

---
## 5 · Why everything happens in $\log_{10}$

This is the single most important design choice in the method, and it is easy to miss.

**The models are trained on $\log_{10}(\text{modulus})$.** So their error is naturally
*multiplicative*, not additive: a model is wrong "by 15%", not "by 12 GPa". A 12 GPa error
is catastrophic on a 20 GPa crystal and negligible on a 400 GPa one.

Sampling in log space encodes that correctly. It also has a property that matters
physically: $10^x > 0$ always, so **a log-space sample can never produce a negative
modulus**, which would be unphysical and would break the square roots in $v_l$ and $v_t$.

The cell below shows what would go wrong if we sampled in GPa instead.

In [ ]:
# Take the softest crystal in the set - where an additive error bar is most dangerous.
soft = pred.loc[pred.K_VRH_pred.idxmin()]
sigma_log = soft.K_VRH_spread_log10
K_pred = soft.K_VRH_pred

# Correct: sample in log space, then exponentiate.
log_space = 10 ** (np.log10(K_pred) + sigma_log * rng.standard_normal(20000))

# Wrong: convert the log spread to an approximate GPa spread and sample additively.
# ln(10) * sigma_log * K is the first-order conversion from log10 spread to GPa spread.
sigma_gpa = math.log(10) * sigma_log * K_pred
gpa_space = K_pred + sigma_gpa * rng.standard_normal(20000)

print(f"softest crystal: {soft.formula}, K = {K_pred:.2f} GPa, log10 spread {sigma_log:.4f}")
print(f"\n  log-space sampling : min {log_space.min():8.3f} GPa   negatives: {(log_space <= 0).sum()}")
print(f"  GPa-space sampling : min {gpa_space.min():8.3f} GPa   negatives: {(gpa_space <= 0).sum()}")
print("\n  A negative modulus has no physical meaning and makes sqrt(G/rho) undefined.")
print("  Log-space sampling makes it impossible by construction.")

---
## 6 · How many samples are enough?

2000 is the project's default. It is not arbitrary — it is where the answer stops moving.

Monte Carlo error falls as $1/\sqrt{n}$, so going from 100 to 10 000 samples cuts the
wobble by a factor of 10. The cell below runs the same crystal at increasing sample counts
and shows where the percentile estimates settle.

In [ ]:
sizes = [50, 100, 250, 500, 1000, 2000, 5000, 20000]
rows = []
for m in sizes:
    r = np.random.RandomState(SEED)          # same seed each time: differences are sample-count only
    kk = 10 ** (np.log10(row.K_VRH_pred) + row.K_VRH_spread_log10 * r.standard_normal(m))
    gg = 10 ** (np.log10(row.G_VRH_pred) + row.G_VRH_spread_log10 * r.standard_normal(m))
    ks = slack_physics(kk, gg, V, RHO, MBAR, N)["kappa_cal"]
    a, b = np.percentile(ks, [5, 95])
    rows.append((m, a, b))
    print(f"  n = {m:6,}   p05 = {a:.5f}   p95 = {b:.5f}")

ref05, ref95 = rows[-1][1], rows[-1][2]      # treat n=20000 as the converged answer
print(f"\n  at the project's n = 2000, p05 is off the n=20000 answer by "
      f"{100*abs(rows[5][1]-ref05)/ref05:.2f}% and p95 by {100*abs(rows[5][2]-ref95)/ref95:.2f}%")
print("  -> 2000 is enough; the remaining wobble is far smaller than the interval itself")

---
## 7 · The catch: this interval is too narrow

Everything above is arithmetically correct and still produces an interval that is
**badly overconfident**. This is the most important thing in the notebook.

`scripts/cgcnn/25_calibration_check.py` tested it against the held-out test set, where the
true DFT moduli are known, and asked: *of the crystals where we claimed 90% confidence, how
many did the interval actually contain?*

In [ ]:
cal = pd.read_csv(os.path.join(ROOT, "results", "cgcnn", "calibration_check.csv"))
print(cal[["nominal", "K_VRH_raw", "K_VRH_recalibrated",
           "G_VRH_raw", "G_VRH_recalibrated"]].to_string(index=False))

r90 = cal[cal.nominal == 0.9].iloc[0]
print(f"\n  at a nominal 90% interval:")
print(f"    K actually covered {100*r90.K_VRH_raw:.1f}% of true values  (should be 90%)")
print(f"    G actually covered {100*r90.G_VRH_raw:.1f}%")
print(f"\n  after rescaling the spread: K {100*r90.K_VRH_recalibrated:.1f}%, "
      f"G {100*r90.G_VRH_recalibrated:.1f}%  -- recovered")

### Why it is too narrow

The Monte Carlo faithfully propagates **the ensemble's disagreement**. But disagreement
between three models is only the *variance* part of the error. The full error is

$$\text{error} = \underbrace{\text{bias}}_{\text{what they all get wrong together}} + \underbrace{\text{variance}}_{\text{what they disagree about}}$$

Three networks with the same architecture, the same training data and the same split
**share their blind spots**. When all three are wrong in the same direction, they agree with
each other — the spread is small — and the Monte Carlo confidently reports a narrow
interval around a wrong answer.

Measured on this project: member disagreement runs **4–6× smaller** than the true error.

**The lesson generalises past this project.** An ensemble's spread is a lower bound on
uncertainty, never the whole of it, and it must be *calibrated against held-out truth*
before it is quoted. Ours was not, until step 25 checked.

---
## 8 · Verification: does our code match the project's?

Everything above was written out longhand. The real pipeline calls one function. If the two
disagree, the explanation above is describing something other than what actually runs — so
let us check, rather than assume.

In [ ]:
# Build the frame monte_carlo_kappa() expects, for the first 200 crystals.
sub = kappa_inputs.head(200).copy()
sub = sub.merge(pred[["material_id", "K_VRH_spread_log10", "G_VRH_spread_log10"]],
                on="material_id", how="left", suffixes=("", "_p"))

theirs = monte_carlo_kappa(sub, n_samples=N_SAMPLES, seed=SEED)   # the project's own function

# Now the same thing, written out by hand exactly as in Section 4.
r = np.random.RandomState(SEED)
m = len(sub)
zk = r.standard_normal((m, N_SAMPLES)); zg = r.standard_normal((m, N_SAMPLES))
lk = np.log10(sub["K_VRH_pred"].values)[:, None] + sub["K_VRH_spread_log10"].values[:, None] * zk
lg = np.log10(sub["G_VRH_pred"].values)[:, None] + sub["G_VRH_spread_log10"].values[:, None] * zg
col = lambda c: sub[c].values[:, None]
mine = slack_physics(10**lk, 10**lg, col("Volume (A3)"), col("Density (g cm-3)"),
                     col("Atomic mass (amu)"), col("Number of Atoms"))["kappa_cal"]
mine_p05 = np.nanpercentile(mine, 5, axis=1)
mine_p95 = np.nanpercentile(mine, 95, axis=1)

d05 = np.abs(mine_p05 - theirs["Kappa_cal_p05"]).max()
d95 = np.abs(mine_p95 - theirs["Kappa_cal_p95"]).max()
print(f"largest disagreement in p05 across 200 crystals: {d05:.3e}")
print(f"largest disagreement in p95 across 200 crystals: {d95:.3e}")
print("\nIDENTICAL" if max(d05, d95) < 1e-12 else "\nDIFFERENT - the explanation above does not match the pipeline")

---
## Summary

| step | what happens | why |
|---|---|---|
| 1 | draw 2000 $\log_{10}K$, $\log_{10}G$ from $\mathcal{N}(\text{prediction}, \text{ensemble spread})$ | log space because the error is multiplicative, and because $10^x$ cannot go negative |
| 2 | push every draw through the **same** `slack_physics` used for the point estimate | no linearisation, no derivatives; the interval is consistent with the central value by construction |
| 3 | report the 5th, 50th and 95th percentiles | the output is skewed even though the input was symmetric, so mean ± 2 sd would be wrong |

**And the caveat that matters more than any of it:** this propagates the ensemble's
*variance* only. It does not see shared bias, so the raw interval covers about **50%** of
true values when it claims 90%. Rescale it against held-out truth before quoting it.